# 2. Chargement des données dans MongoDB

Objectif : charger le jeu de données nettoyé (`data/processed/cyberbullying_clean.csv`,
produit par le notebook 01) dans une collection MongoDB, pour pouvoir ensuite
l'interroger depuis l'API et les notebooks suivants.

**Prérequis :** une instance MongoDB doit être démarrée, par exemple avec :
```bash
cd docker
docker-compose up -d mongodb
```

## Connexion à MongoDB

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from pymongo import ASCENDING, MongoClient
from pymongo.errors import ServerSelectionTimeoutError

load_dotenv("../.env")

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
MONGO_DB_NAME = os.getenv("MONGO_DB_NAME", "cyberbullying_db")
MONGO_COLLECTION_PROCESSED = os.getenv("MONGO_COLLECTION_PROCESSED", "messages_processed")

print(f"URI          : {MONGO_URI}")
print(f"Base         : {MONGO_DB_NAME}")
print(f"Collection   : {MONGO_COLLECTION_PROCESSED}")

URI          : mongodb://localhost:27017
Base         : cyberbullying_db
Collection   : messages_processed


In [2]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)

try:
    client.admin.command("ping")
    print("Connexion à MongoDB réussie.")
except ServerSelectionTimeoutError as exc:
    raise RuntimeError(
        "Impossible de joindre MongoDB. Démarrez une instance avant de continuer, "
        "par ex. : `cd docker && docker-compose up -d mongodb`"
    ) from exc

db = client[MONGO_DB_NAME]
collection = db[MONGO_COLLECTION_PROCESSED]

Connexion à MongoDB réussie.


## Chargement des données nettoyées

In [3]:
PROCESSED_PATH = "../data/processed/cyberbullying_clean.csv"

df = pd.read_csv(PROCESSED_PATH)
print(f"Lignes chargées : {len(df)}")
df.head()

Lignes chargées : 5740


,Text,text_clean,Label,Types,text_len
0,Ten outside soon doctor shake everyone treatme...,ten outside soon doctor shake everyone treatme...,Not-Bullying,Aucun,10
1,my life has come to a standstill and at this p...,my life has come to a standstill and at this p...,Not-Bullying,Aucun,11
2,girl this nigga make me sick to my stomach,girl this nigga make me sick to my stomach,Bullying,Ethnicity,9
3,I wanna fuck you,i wanna fuck you,Bullying,Sexual,4
4,"Oh hey, you should be ashamed of your disgusti...",oh hey you should be ashamed of your disgustin...,Not-Bullying,Aucun,10


In [4]:
records = df.to_dict(orient="records")

# Idempotent : on vide la collection avant de réinsérer, pour pouvoir relancer
# ce notebook plusieurs fois sans dupliquer les documents.
deleted = collection.delete_many({})
print(f"Documents existants supprimés : {deleted.deleted_count}")

result = collection.insert_many(records)
print(f"Documents insérés : {len(result.inserted_ids)}")

Documents existants supprimés : 0
Documents insérés : 5740


In [5]:
# Index pour accélérer les requêtes/agrégations par classe et par sous-type
collection.create_index([("Label", ASCENDING)])
collection.create_index([("Types", ASCENDING)])

print("Index existants :")
for name, info in collection.index_information().items():
    print(f"  {name}: {info['key']}")

Index existants :
  _id_: [('_id', 1)]
  Label_1: [('Label', 1)]
  Types_1: [('Types', 1)]


## Vérification de l'insertion

In [6]:
total = collection.count_documents({})
print(f"Nombre total de documents dans '{MONGO_COLLECTION_PROCESSED}' : {total}")

assert total == len(df), "Le nombre de documents ne correspond pas au CSV source !"
print("OK : le nombre de documents correspond au fichier CSV source.")

Nombre total de documents dans 'messages_processed' : 5740
OK : le nombre de documents correspond au fichier CSV source.


In [7]:
# Aperçu de quelques documents insérés
for doc in collection.find().limit(3):
    print(doc)

{'_id': ObjectId('6a99548e399684f703c6f902'), 'Text': 'Ten outside soon doctor shake everyone treatment seat radio land.', 'text_clean': 'ten outside soon doctor shake everyone treatment seat radio land', 'Label': 'Not-Bullying', 'Types': 'Aucun', 'text_len': 10}
{'_id': ObjectId('6a99548e399684f703c6f903'), 'Text': 'my life has come to a standstill and at this point.', 'text_clean': 'my life has come to a standstill and at this point', 'Label': 'Not-Bullying', 'Types': 'Aucun', 'text_len': 11}
{'_id': ObjectId('6a99548e399684f703c6f904'), 'Text': 'girl this nigga make me sick to my stomach', 'text_clean': 'girl this nigga make me sick to my stomach', 'Label': 'Bullying', 'Types': 'Ethnicity', 'text_len': 9}


In [8]:
# Vérification de la répartition des classes directement via une agrégation MongoDB
pipeline = [
    {"$group": {"_id": "$Label", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
]

for row in collection.aggregate(pipeline):
    print(row)

{'_id': 'Bullying', 'count': 3140}
{'_id': 'Not-Bullying', 'count': 2600}
